# Bagré | 02: Land surface temperature and vegetation
    **Goal:** distinguish the temperature of the land surface from air temperature. Cooler vegetated surfaces may reflect evaporative cooling, but temperature also changes with weather, cover, soil and observation time. NDVI describes greenness; NDMI is sensitive to canopy water content and structure, not a direct soil-moisture measurement.

## 1. Connect Drive and Earth Engine
    Use a CPU runtime. Run this cell after every runtime restart. You need a registered Earth Engine Google Cloud project (not your Gmail address). Set its project ID below. The Drive folder must be in your My Drive, or a shortcut with the same name.

In [ ]:
%pip -q install earthengine-api rasterio pyproj
import ee
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Bagre_ET_Water_Stress')
assert ROOT.is_dir(), f'Open Drive and confirm this folder exists: {ROOT}'
for sub in ['data', 'outputs', 'figures', 'qgis']:
    (ROOT / sub).mkdir(exist_ok=True)
PROJECT = 'YOUR_EARTH_ENGINE_PROJECT_ID'  # CHANGE THIS
assert PROJECT != 'YOUR_EARTH_ENGINE_PROJECT_ID', 'Set your registered Cloud project ID.'
try:
    ee.Initialize(project=PROJECT)
except ee.EEException:
    ee.Authenticate()
    ee.Initialize(project=PROJECT)
print('Earth Engine connection:', ee.Number(1).getInfo())

## 2. Study design
    Reuse the approximate Bagré pilot window from the earlier water project. It includes agricultural land and other land covers; it is not an official irrigation boundary. January–April 2021 is a single dry-season demonstration. WorldCover 2021 cropland is a proxy, not a field survey. All end dates are exclusive. Do not change dates or AOI halfway through the series.

In [ ]:
import calendar
from datetime import date, timedelta
AOI_BOUNDS = [-0.59, 11.41, -0.47, 11.52]  # west, south, east, north
AOI = ee.Geometry.Rectangle(AOI_BOUNDS, proj='EPSG:4326', geodesic=False)
CENTER = AOI.centroid(1)
CRS = 'EPSG:32630'
MONTHS = [(f'2021-{m:02d}-01', f'2021-{m+1:02d}-01') for m in range(1, 5)]
S2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED').filterBounds(AOI)
L8 = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').filterBounds(AOI)
      .filter(ee.Filter.eq('PROCESSING_LEVEL', 'L2SP')))
MOD = ee.ImageCollection('MODIS/061/MOD16A2GF').filterBounds(AOI)
WAP = ee.ImageCollection('FAO/WAPOR/3/L1_AETI_D').filterBounds(AOI)
ERA = ee.ImageCollection('ECMWF/ERA5_LAND/DAILY_AGGR').filterBounds(AOI)
WC = ee.ImageCollection('ESA/WorldCover/v200').first().select('Map')

### Download helper
    We download only the small pilot as GeoTIFF, with explicit CRS and scale. Earth Engine performs the processing remotely. Native thermal resolution remains about 100 m; a finer output grid would not create new information. The code checks that downloaded files have the expected number of bands and a valid coordinate system. Temporary files prevent incomplete downloads from masquerading as results.

In [ ]:
import requests
import rasterio
def download(image, path, scale, bands):
    path = Path(path)
    tmp = path.with_suffix('.partial.tif')
    url = image.select(bands).toFloat().clip(AOI).unmask(-9999, False).getDownloadURL({
        'name': path.stem, 'region': AOI, 'scale': scale,
        'crs': CRS, 'format': 'GEO_TIFF', 'filePerBand': False
    })
    with requests.get(url, stream=True, timeout=180) as response:
        response.raise_for_status()
        with tmp.open('wb') as f:
            for chunk in response.iter_content(1024 * 1024):
                f.write(chunk)
    with rasterio.open(tmp, 'r+') as src:
        assert src.count == len(bands) and src.crs.to_epsg() == 32630
        src.nodata = -9999
        for i, band in enumerate(bands, 1):
            src.set_band_description(i, band)
        print(path.name, src.shape, list(src.descriptions))
    tmp.replace(path)
    return path

## 3. Cloud and quality masking
    Landsat L2SP already contains a surface-temperature retrieval. Convert the stored DN to Kelvin using the official scale and offset, then subtract 273.15. Exclude fill, dilated cloud, cirrus, cloud, cloud shadow and snow (QA_PIXEL bits 0–5), saturated pixels, and estimated ST uncertainty above 3 K (selected after sensitivity testing). Sentinel-2 uses only SCL classes 4, 5 and 6 (vegetation, bare ground, water); clouds, uncertain classes and shadows are excluded. This is a conservative mask, not perfect cloud detection.

In [ ]:
def prep_l8(im):
    qa = im.select('QA_PIXEL')
    clear = qa.bitwiseAnd(63).eq(0).And(im.select('QA_RADSAT').eq(0))
    clear = clear.And(im.select('ST_QA').multiply(0.01).lte(3))
    lst = im.select('ST_B10').multiply(0.00341802).add(149).subtract(273.15)
    return lst.rename('lst_c').updateMask(clear).copyProperties(im, ['system:time_start'])
def prep_s2(im):
    scl = im.select('SCL')
    clear = scl.eq(4).Or(scl.eq(5)).Or(scl.eq(6))
    sr = im.select(['B4','B8','B11']).multiply(0.0001).updateMask(clear)
    red, nir, swir = sr.select('B4'), sr.select('B8'), sr.select('B11')
    ndvi = nir.subtract(red).divide(nir.add(red)).updateMask(nir.add(red).gt(0))
    ndmi = nir.subtract(swir).divide(nir.add(swir)).updateMask(nir.add(swir).gt(0))
    return ndvi.rename('ndvi').addBands(ndmi.rename('ndmi')).copyProperties(im, ['system:time_start'])

## 4. Monthly composites
    Median is robust to individual clear-scene outliers. Export the median and count of valid observations; a month with no usable observation at a location remains missing. Sentinel-2 indices and Landsat temperature are separate monthly composites, not same-day measurements.

In [ ]:
for start, end in MONTHS:
    month = start[:7]
    l = L8.filterDate(start, end).map(prep_l8)
    s = S2.filterDate(start, end).map(prep_s2)
    assert l.size().getInfo() > 0 and s.size().getInfo() > 0
    li = l.median().addBands(l.count().rename('lst_count'))
    si = s.median().addBands(s.select('ndvi').count().rename('ndvi_count'))
    download(li, ROOT/'data'/f'lst_{month}.tif', 100, ['lst_c','lst_count'])
    download(si, ROOT/'data'/f'veg_{month}.tif', 20, ['ndvi','ndmi','ndvi_count'])

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(15, 10), constrained_layout=True)
for j, (start, _) in enumerate(MONTHS):
    month = start[:7]
    with rasterio.open(ROOT/'data'/f'lst_{month}.tif') as src:
        lst = src.read(1, masked=True)
        count = src.read(2, masked=True)
    with rasterio.open(ROOT/'data'/f'veg_{month}.tif') as src:
        ndvi = src.read(1, masked=True)
        ndmi = src.read(2, masked=True)
    for i, (a, label, cmap, lo, hi) in enumerate([
        (lst, 'LST °C', 'inferno', 20, 50),
        (ndvi, 'NDVI', 'RdYlGn', 0, 0.8),
        (ndmi, 'NDMI', 'BrBG', -0.4, 0.5)]):
        p = axes[i,j].imshow(a, cmap=cmap, vmin=lo, vmax=hi)
        axes[i,j].set_title(f'{month} {label}')
        axes[i,j].axis('off')
        fig.colorbar(p, ax=axes[i,j], shrink=.7)
    print(month, 'LST valid fraction:', float(lst.count()/lst.size),
          'median valid LST observations:', float(np.ma.median(count)))
fig.savefig(ROOT/'figures'/'monthly_temperature_vegetation.png', dpi=160)
plt.show()

In [ ]:
for start, end in MONTHS:
    month = start[:7]

    with rasterio.open(ROOT / 'data' / f'lst_{month}.tif') as src:
        lst = src.read(1, masked=True)

    print(
        month,
        f"Valid pixels: {lst.count():,}",
        f"Coverage: {100 * lst.count() / lst.size:.2f}%"
    )

In [ ]:
for start, end in MONTHS:
    collection = L8.filterDate(start, end)

    print("\nMonth:", start[:7])

    for threshold in [2, 3, 5, 10, None]:

        def diagnostic_mask(im):
            clear = im.select('QA_PIXEL').bitwiseAnd(63).eq(0)
            clear = clear.And(im.select('QA_RADSAT').eq(0))

            if threshold is not None:
                clear = clear.And(
                    im.select('ST_QA').multiply(0.01).lte(threshold)
                )

            return im.select('ST_B10').updateMask(clear)

        composite = collection.map(diagnostic_mask).median()

        coverage = composite.mask().unmask(0).reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=AOI,
            scale=100,
            crs=CRS,
            maxPixels=1e7
        ).getInfo().get('ST_B10')

        label = f"{threshold} K" if threshold else "No ST_QA filter"

        print(label, f"{100 * coverage:.2f}%" if coverage is not None else "No data")

**Checkpoint:** compare January and April. Where is vegetation greener? Are missing values extensive? Do not interpret warm regions as water-stressed without controlling for cover and weather. If a month has little coverage, retain the missingness and discuss it.